# InfraVision: finish training the thermal YOLO model

Continues `runs/thermal/baseline` from **epoch 7 to 50** using the saved `last.pt`
(model weights, optimizer state and learning-rate schedule are all kept).

**Before running:**
1. Upload `thermal_yolo_colab.zip` to Google Drive, into a folder named `Infravision`
   (so the path is `My Drive/Infravision/thermal_yolo_colab.zip`).
2. In Colab: **Runtime → Change runtime type → T4 GPU**.
3. Run the cells from top to bottom.

Training is saved to Google Drive after every epoch. If Colab disconnects,
reconnect and run all cells again: it picks up from the last saved epoch.

## 1. Check the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch
assert torch.cuda.is_available(), "No GPU. Use Runtime -> Change runtime type -> T4 GPU, then rerun."
print("GPU ready:", torch.cuda.get_device_name(0))

## 2. Install the same Ultralytics version used on the laptop

In [ ]:
!pip install -q ultralytics==8.4.177
import ultralytics
print("ultralytics", ultralytics.__version__)

## 3. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/Infravision")
ZIP_PATH = DRIVE_DIR / "thermal_yolo_colab.zip"
RUN_DIR = DRIVE_DIR / "runs" / "thermal_baseline_resumed"   # saved on Drive, survives disconnects
DATA_YAML = Path("/content/thermal_person/data.yaml")

assert ZIP_PATH.exists(), f"Zip not found: {ZIP_PATH}. Upload it to My Drive/Infravision/ first."
print("Zip  :", ZIP_PATH)
print("Runs :", RUN_DIR)

## 4. Unpack the dataset and checkpoint onto the Colab disk (fast local storage)

In [ ]:
import shutil, zipfile

if not DATA_YAML.exists():
    local_zip = Path("/content/thermal_yolo_colab.zip")
    shutil.copy(ZIP_PATH, local_zip)          # reading from local disk is much faster than from Drive
    with zipfile.ZipFile(local_zip) as z:
        z.extractall("/content")
    local_zip.unlink()

for split in ("train", "val", "test"):
    n_img = len(list(Path(f"/content/thermal_person/images/{split}").iterdir()))
    n_lbl = len(list(Path(f"/content/thermal_person/labels/{split}").iterdir()))
    print(f"{split:5s}: {n_img} images, {n_lbl} labels")
print(DATA_YAML.read_text())

## 5. Continue training (epoch 7 → 50)

Uses the Drive copy of `last.pt` if a previous Colab session already trained some epochs,
otherwise the original `last.pt` from the laptop (stopped after epoch 6).
Early stopping (`patience=10`) may end training before epoch 50 if validation stops improving.
Expect roughly 1–2 minutes per epoch on a T4.

In [ ]:
from ultralytics import YOLO

drive_last = RUN_DIR / "weights" / "last.pt"
ckpt = drive_last if drive_last.exists() else Path("/content/weights/last.pt")
print("Resuming from:", ckpt)

model = YOLO(str(ckpt))
try:
    model.train(
        resume=True,
        data=str(DATA_YAML),     # replaces the old Windows path stored in the checkpoint
        save_dir=str(RUN_DIR),   # write results to Google Drive
        device=0,
        batch=16,                # T4 has room for 16 at 640 px (laptop used 8)
        workers=2,
    )
except AssertionError as e:
    # Raised when the checkpoint already finished all epochs
    print("Nothing to resume:", e)

## 6. Evaluate the new model on the test split (images never used for training)

In [ ]:
best = RUN_DIR / "weights" / "best.pt"
metrics = YOLO(str(best)).val(data=str(DATA_YAML), split="test", imgsz=640, batch=16, device=0)

print()
print(f"Test precision : {metrics.box.mp:.3f}")
print(f"Test recall    : {metrics.box.mr:.3f}")
print(f"Test mAP50     : {metrics.box.map50:.3f}")
print(f"Test mAP50-95  : {metrics.box.map:.3f}")
print()
print("For reference, the laptop model at epoch 6 scored mAP50 = 0.637 on the val split.")

## 7. Training curve

In [ ]:
import pandas as pd
df = pd.read_csv(RUN_DIR / "results.csv")
df.columns = df.columns.str.strip()
display(df[["epoch", "metrics/precision(B)", "metrics/recall(B)", "metrics/mAP50(B)", "metrics/mAP50-95(B)"]].tail(10))
df.plot(x="epoch", y=["metrics/mAP50(B)", "metrics/recall(B)", "metrics/precision(B)"], grid=True, figsize=(8, 4));

## 8. Download the results

Downloads `thermal_baseline_resumed.zip` (weights, `results.csv`, plots).
On the laptop, put `weights/best.pt` into
`human_detection/runs/thermal/baseline/weights/best.pt` (keep a copy of the old one first),
or point `YOLO_MODEL_PATH` in `integrate_infravision.py` at the new file.
The same files also stay in Google Drive under `My Drive/Infravision/runs/`.

In [ ]:
from google.colab import files
archive = shutil.make_archive("/content/thermal_baseline_resumed", "zip", RUN_DIR)
files.download(archive)